In [73]:
import pandas as pd 
import numpy as np

customers=pd.read_csv("customers.csv")
products=pd.read_csv("products.csv")
sales=pd.read_csv("sales.csv")
sales["Coupon_Code"] = sales["Coupon_Code"].fillna("No Coupon Code")
sales["New_Rating"]=sales.groupby("Product_ID")["Rating"].transform("mean").round().astype(int)
sales["Rating"]=sales["Rating"].fillna(0)
sales.loc[sales["Rating"] == 0,"Rating"] = sales["New_Rating"]
sales.drop("New_Rating", axis=1, inplace=True)
sales["Review_Text"]=sales["Review_Text"].fillna("Not Reviewed")
x=sales.copy()
x["Valid_Sales"] = (x["Total_Amount"].where(x["Order_Status"].isin(["Delivered", "Processing", "Shipped"]), 0).groupby(x["Customer_ID"]).transform("sum"))
x["Corrupted_Sales"] = (x["Total_Amount"].where(x["Order_Status"].isin(["Cancelled","Returned"]), 0).groupby(x["Customer_ID"]).transform("sum"))
x["Total_Sales"]=x["Valid_Sales"]+x["Corrupted_Sales"]
x["Valid_Orders"] = x["Order_Status"].isin(["Delivered", "Processing", "Shipped"]).groupby(x["Customer_ID"]).transform("sum")
x["Corrupted_Orders"] = x["Order_Status"].isin(["Cancelled", "Returned"]).groupby(x["Customer_ID"]).transform("sum")
x["Total_Orders_Count"]=x["Corrupted_Orders"]+x["Valid_Orders"]
x


,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Time,Delivery_Date,Quantity,Unit_Price,Order_Value,Shipping_Cost,...,City,State,Customer_Age,Customer_Age_Group,Valid_Sales,Corrupted_Sales,Total_Sales,Valid_Orders,Corrupted_Orders,Total_Orders_Count
0,ORD0000000001,CUST00014303,PROD001017,2026-06-07,08:20:00,2026-06-10,1,2622.46,2622.46,0.0,...,Chennai,Tamil Nadu,43,36-45,319715.47,84528.35,404243.82,5,2,7
1,ORD0000000002,CUST00034253,PROD000703,2025-02-04,20:05:00,2025-02-07,1,90258.00,90258.00,0.0,...,Durgapur,West Bengal,22,18-25,218963.05,0.00,218963.05,3,0,3
2,ORD0000000003,CUST00000421,PROD000318,2026-03-12,14:59:00,2026-03-19,1,3536.50,3536.50,0.0,...,Patiala,Punjab,41,36-45,207525.23,0.00,207525.23,11,0,11
3,ORD0000000004,CUST00026507,PROD000213,2025-01-27,14:33:00,2025-02-02,1,34081.56,34081.56,0.0,...,Rajkot,Gujarat,29,26-35,475738.74,0.00,475738.74,9,0,9
4,ORD0000000005,CUST00009418,PROD001437,2024-06-30,07:19:00,2024-07-02,2,69021.17,138042.34,0.0,...,Surat,Gujarat,39,36-45,539781.79,62216.23,601998.02,5,2,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,ORD0000249996,CUST00008137,PROD001818,2024-10-06,19:17:00,2024-10-08,1,1827.96,1827.96,0.0,...,Sohna,Haryana,37,36-45,36740.24,0.00,36740.24,5,0,5
249996,ORD0000249997,CUST00027003,PROD000955,2024-08-28,05:26:00,2024-08-31,1,4040.86,4040.86,0.0,...,Mangalore,Karnataka,34,26-35,24897.06,0.00,24897.06,3,0,3
249997,ORD0000249998,CUST00027511,PROD001894,2025-12-09,13:44:00,2025-12-12,2,10516.60,21033.20,0.0,...,Delhi NCR,Delhi,21,18-25,55424.43,5043.74,60468.17,4,1,5
249998,ORD0000249999,CUST00022175,PROD001950,2025-05-01,18:24:00,2025-05-05,1,7847.77,7847.77,0.0,...,Varanasi,UP,26,26-35,111430.68,85424.32,196855.00,5,3,8


In [83]:
x=x[["Customer_ID","Order_Date","Delivery_Date","Valid_Sales","Corrupted_Sales","Total_Sales","Valid_Orders","Corrupted_Orders","Total_Orders_Count"]]
x = x.drop_duplicates(subset=['Customer_ID'])
x=pd.merge(x,customers,on="Customer_ID",how="left")
x = x.drop(columns=["Customer_Name","Date_of_Birth","Email","Phone","Pincode","Registration_Date"])
x = x.sort_values(by="Customer_ID").reset_index(drop=True)
x.to_csv("Customer-Sales-Analysis.csv", index=False)
